In [1]:
import os
import cv2
import xml.etree.ElementTree as ET
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization, GlobalAveragePooling2D, Add, Reshape
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.model_selection import train_test_split

In [2]:
# --- Paths ---
DATASET_PATH = 'D:/CSP/Dataset/VOC2028/VOC2028'
ANNOTATIONS_PATH = os.path.join(DATASET_PATH, "Annotations")
IMAGES_PATH = os.path.join(DATASET_PATH, "JPEGImages")

# --- Labels ---
LABELS = {"hat": 1, "person": 0}  # Helmet is positive (1), No Helmet is negative (0)

In [3]:
# --- Function to parse annotation XML files ---
def parse_voc_annotation(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    for obj in root.findall("object"):
        label = obj.find("name").text.lower().strip()
        if label in LABELS:
            return LABELS[label]
    return None

In [4]:
# --- Load all annotations dynamically ---
annotations = []
for filename in os.listdir(ANNOTATIONS_PATH):
    xml_file = os.path.join(ANNOTATIONS_PATH, filename)
    img_file = os.path.join(IMAGES_PATH, filename.replace(".xml", ".jpg"))
    
    if os.path.exists(img_file):
        label = parse_voc_annotation(xml_file)
        if label is not None:
            annotations.append((img_file, label))

print(f"Loaded {len(annotations)} annotated images")


Loaded 7581 annotated images


In [5]:
# --- Split dataset into training & testing ---
train_files, test_files = train_test_split(annotations, test_size=0.2, random_state=42)

IMG_SIZE = 224
BATCH_SIZE = 32

In [6]:
# --- Function to load and preprocess images dynamically ---
def load_and_preprocess(image_path, label):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0  # Normalize
    return img, label

In [7]:

# --- Convert dataset to tf.data format ---
def prepare_dataset(file_label_list, batch_size):
    image_paths, labels = zip(*file_label_list)
    dataset = tf.data.Dataset.from_tensor_slices((list(image_paths), list(labels)))
    dataset = dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.shuffle(buffer_size=1000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset


In [8]:
# --- Create training and validation datasets ---
train_dataset = prepare_dataset(train_files, BATCH_SIZE)
test_dataset = prepare_dataset(test_files, BATCH_SIZE)

In [9]:
# --- CNN Backbone for Feature Extraction ---
def build_cnn_backbone(inputs):
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)

    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)

    x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)
    
    return x

In [10]:
# --- Transformer Encoder Block ---
def transformer_block(inputs, num_heads, dff, dropout_rate=0.1):
    x = LayerNormalization(epsilon=1e-6)(inputs)
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=inputs.shape[-1])(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    out1 = Add()([inputs, attn_output])

    x = LayerNormalization(epsilon=1e-6)(out1)
    ffn_output = Dense(dff, activation='relu')(x)
    ffn_output = Dense(inputs.shape[-1])(ffn_output)
    ffn_output = Dropout(dropout_rate)(ffn_output)
    
    out2 = Add()([out1, ffn_output])
    return out2


In [11]:
# --- Full CNN + Transformer Model ---
def build_model(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_heads=4, dff=256, num_transformer_blocks=2):
    inputs = Input(shape=input_shape)

    # CNN Backbone
    cnn_features = build_cnn_backbone(inputs)

    # Reshape for Transformer
    patches = Reshape((cnn_features.shape[1] * cnn_features.shape[2], cnn_features.shape[3]))(cnn_features)

    # Transformer Layers
    for _ in range(num_transformer_blocks):
        patches = transformer_block(patches, num_heads, dff)

    # Global Average Pooling & Classification
    x = GlobalAveragePooling2D()(cnn_features)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])
    
    return model

In [12]:
# --- Build Optimized CNN + Transformer Model ---
model = build_model(num_heads=4, dff=256, num_transformer_blocks=2)

# --- Early Stopping & Learning Rate Scheduler ---
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

In [13]:
# --- Model Training ---
history = model.fit(
    train_dataset,
    epochs=25,
    validation_data=test_dataset,
    callbacks=[early_stopping, lr_scheduler],
    batch_size=BATCH_SIZE
)


Epoch 1/25
190/190 ━━━━━━━━━━━━━━━━━━━━ 544s 3s/step - accuracy: 0.8000 - loss: 0.4364 - val_accuracy: 0.4450 - val_loss: 3.1992 - learning_rate: 5.0000e-04
Epoch 2/25
190/190 ━━━━━━━━━━━━━━━━━━━━ 508s 3s/step - accuracy: 0.8602 - loss: 0.3071 - val_accuracy: 0.4450 - val_loss: 3.4232 - learning_rate: 5.0000e-04
Epoch 3/25
190/190 ━━━━━━━━━━━━━━━━━━━━ 2496s 13s/step - accuracy: 0.8915 - loss: 0.2444 - val_accuracy: 0.4469 - val_loss: 1.5212 - learning_rate: 5.0000e-04
Epoch 4/25
190/190 ━━━━━━━━━━━━━━━━━━━━ 516s 3s/step - accuracy: 0.9128 - loss: 0.2149 - val_accuracy: 0.4542 - val_loss: 2.0007 - learning_rate: 5.0000e-04
Epoch 5/25
190/190 ━━━━━━━━━━━━━━━━━━━━ 516s 3s/step - accuracy: 0.9261 - loss: 0.1836 - val_accuracy: 0.8708 - val_loss: 0.2957 - learning_rate: 5.0000e-04
Epoch 6/25
190/190 ━━━━━━━━━━━━━━━━━━━━ 518s 3s/step - accuracy: 0.9418 - loss: 0.1554 - val_accuracy: 0.4483 - val_loss: 2.6239 - learning_rate: 5.0000e-04
Epoch 7/25
190/190 ━━━━━━━━━━━━━━━━━━━━ 516s 3s/step - a

In [14]:
# --- Evaluate Model ---
test_loss, test_acc = model.evaluate(test_dataset)
print(f"Test Accuracy: {test_acc * 100:.2f}%")


48/48 ━━━━━━━━━━━━━━━━━━━━ 19s 368ms/step - accuracy: 0.9598 - loss: 0.1054
Test Accuracy: 96.44%


In [15]:

# --- Save Model ---
model.save("cnn_transformer_helmet_detection_model.h5")

In [16]:
model.save("cnn_transformer_helmet_detection_model.keras")

In [24]:
 # Image path for testing (replace with your image path)
TEST_IMAGE_PATH = r"D:\testimage.jpg"
# Verify if the image exists
if not os.path.exists(TEST_IMAGE_PATH):
    print(f"Error: The file '{TEST_IMAGE_PATH}' does not exist. Check the path!")
else:
    print(f"Image found at: {TEST_IMAGE_PATH}")


# Image size (same as used during training)
IMG_SIZE = 224

# Load and preprocess the test image
def preprocess_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0  # Resize and normalize
    img = np.expand_dims(img, axis=0)  # Add batch dimension
    return img

# Preprocess the test image
input_image = preprocess_image(TEST_IMAGE_PATH)

Image found at: D:\testimage.jpg


In [25]:
# Make prediction
prediction = model.predict(input_image)

# Convert prediction to class label
if prediction[0][0] > 0.5:
    result = "Helmet Detected"
else:
    result = "No Helmet Detected"

# Show result
print(f"Prediction: {result} (Confidence: {prediction[0][0]*100:.2f}%)")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step
Prediction: No Helmet Detected (Confidence: 3.61%)


In [27]:
# Directory with multiple images
TEST_IMAGE_DIR = r"D:/CSP/TestImages"

# Loop through all images in the directory
for filename in os.listdir(TEST_IMAGE_DIR):
    if filename.endswith(".jpg") or filename.endswith(".png"):
        image_path = os.path.join(TEST_IMAGE_DIR, filename)
        input_image = preprocess_image(image_path)
        
        # Predict and print results
        prediction = model.predict(input_image)
        result = "Helmet Detected" if prediction[0][0] > 0.5 else "No Helmet Detected"
        print(f"{filename}: {result} (Confidence: {prediction[0][0]*100:.2f}%)")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
000075.jpg: Helmet Detected (Confidence: 97.64%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
000076.jpg: Helmet Detected (Confidence: 98.63%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
000077.jpg: Helmet Detected (Confidence: 100.00%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
000078.jpg: Helmet Detected (Confidence: 99.71%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
000079.jpg: Helmet Detected (Confidence: 99.99%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
000080.jpg: Helmet Detected (Confidence: 99.99%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
PartB_00445.jpg: No Helmet Detected (Confidence: 0.00%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
PartB_00446.jpg: No Helmet Detected (Confidence: 1.69%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
PartB_00447.jpg: No Helmet Detected (Confidence: 0.30%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
PartB_00448.jpg: No Helmet Detected (Confidence: 1.84%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
PartB_00449.jpg: No Helmet Detected (Confidence: 0.00%)
1/1 ━━━